# Does CCM library length change lag variability?
Compare three library lengths on the existing linear, nonlinear and independent
original uniform records from experiment 02. This small diagnostic changes only
library length: nine scans, 100 random libraries per scan, and both directions.

The experiment measures random-library variability conditional on one record per
process form. It does not run surrogate tests, update causal decisions, or select
production settings. Run from the project root with the pinned scientific
environment and the optional notebook tools in `agent/requirements-notebook.txt`.


## 1. Keep the comparison small and explicit
Use the current E=3 embedding, 20-year spacing, temporal exclusion and 31 target
shifts. Sequential CPU calls reuse native PyEDM objects; no GPU backend or new
parallel implementation is needed for these nine scans.


In [ ]:
import os
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS"]:
    os.environ[name] = "1"

from pathlib import Path
from importlib.metadata import version
import json
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyEDM
from IPython.display import display
from threadpoolctl import threadpool_limits
from toolbox.causal import _ccm_lag_peak

PROCESSES = ["linear", "nonlinear", "independent"]  # One existing history per form.
LIBRARY_SIZES = [350, 700, 1119]  # Numbers of embedded states, not time durations.
N_LIBRARIES = 100               # Random draws per library length, without replacement.
LIBRARY_SEED = 20261309         # Same master seed as the production lag screen.
E = 3                          # Embedding dimension.
TAU = -1                       # Embedding spacing in 20-year steps; past history.
EXCLUSION_RADIUS = 10          # Exclude neighbors within 10 steps (200 years).
LAG_STEPS = np.arange(-15, 16)  # Target shifts: -300 to +300 years, in 20-year steps.
DT_KYR = 0.020                 # Original sampling interval, kyr.
DIRECTIONS = ["X -> Y", "Y -> X"]  # Causal hypotheses, not cross-map orientation.
COLORS = ["#4682B4", "#E69F00", "#009E73"]

ROOT = Path.cwd()
assert (ROOT / "toolbox/causal.py").is_file(), "Run from the project root."
INPUT_DIR = ROOT / "data/processed/02_check_causal_pipeline"
OUTPUT_DIR = ROOT / "data/processed/02_6_check_ccm_library_length"
FIGURE_DIR = ROOT / "figures/02_6_check_ccm_library_length"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
assert version("pyEDM") == "2.5.7", "This diagnostic uses the verified PyEDM 2.5.7 path."
assert N_LIBRARIES >= 2 and E >= 1 and TAU < 0
assert np.all(np.diff(LAG_STEPS) > 0) and LAG_STEPS[0] < 0 < LAG_STEPS[-1]
assert 0 in LAG_STEPS

from toolbox.plotting import add_panel_labels


## 2. Reuse the data and common anchors
Read the saved 29–1 kyr BP inputs without regenerating, interpolating or smoothing
them. All shifts use the same candidate rows; the 100 seed values are reused across
shifts at each library length, while different lengths need not give nested subsets.


In [ ]:
series = {}
input_rows = []
for process in PROCESSES:
    path = INPUT_DIR / f"r0_{process}_original_uniform_series.csv"
    frame = pd.read_csv(path)  # Preserve the production estimator input-loading path.
    assert len(frame) == 1401
    assert np.isfinite(frame[["time", "age", "x", "y"]]).all().all()
    np.testing.assert_allclose(np.diff(frame["time"]), DT_KYR, rtol=0, atol=1e-10)
    np.testing.assert_allclose(np.diff(frame["age"]), -DT_KYR, rtol=0, atol=1e-10)
    np.testing.assert_allclose(frame["age"].iloc[[0, -1]], [29, 1])
    assert (frame[["x", "y"]].max() > frame[["x", "y"]].min()).all()
    series[process] = frame
    input_rows.append({"process": process, "points": len(frame),
                       "oldest_age_kyr_BP": frame["age"].iloc[0],
                       "youngest_age_kyr_BP": frame["age"].iloc[-1],
                       "first_x": frame["x"].iloc[0], "first_y": frame["y"].iloc[0]})

n_points = len(series[PROCESSES[0]])
history = (E - 1) * abs(TAU)
anchors = np.arange(max(history, -int(LAG_STEPS[0])), n_points - int(LAG_STEPS[-1]))
valid_lib = np.zeros(n_points, dtype=bool)
valid_lib[anchors] = True
assert all(E + 2 <= size <= len(anchors) for size in LIBRARY_SIZES)
library_seeds = np.random.default_rng(LIBRARY_SEED).integers(
    1, 2**31 - 1, size=N_LIBRARIES
)
lag_years = LAG_STEPS * DT_KYR * 1000
display(pd.DataFrame(input_rows))
display(pd.DataFrame({"library_size": LIBRARY_SIZES,
                      "fraction_of_common_anchors": np.array(LIBRARY_SIZES) / len(anchors)}))
print(f"Common anchors: {anchors[0]}..{anchors[-1]} ({len(anchors)} rows); "
      f"{len(np.unique(library_seeds))}/{N_LIBRARIES} distinct seed values.")


## 3. Retain every native lag curve
For X -> Y, reconstruct X(t + Tp) from Y's embedding; the reverse map swaps X and Y.
At fixed length and seed, PyEDM draws the same library at every shift and predicts
all common anchors: this is not an 80/20 training/test split.


In [ ]:
# Dimensions: process, library length, random library, target shift, direction.
lag_rho = np.full((len(PROCESSES), len(LIBRARY_SIZES), N_LIBRARIES,
                   len(LAG_STEPS), len(DIRECTIONS)), np.nan)
errors = []
timings = []
with threadpool_limits(limits=1):
    for process_index, process in enumerate(PROCESSES):
        data = series[process][["time", "x", "y"]]
        for size_index, library_size in enumerate(LIBRARY_SIZES):
            started = time.perf_counter()
            for lag_index, lag in enumerate(LAG_STEPS):
                model = None
                for draw, seed in enumerate(library_seeds):
                    try:
                        if model is None:
                            model = pyEDM.CCM(
                                dataFrame=data, columns="y", target="x",
                                E=E, tau=TAU, Tp=int(lag),
                                exclusionRadius=EXCLUSION_RADIUS,
                                libSizes=[library_size], sample=1, seed=int(seed),
                                validLib=valid_lib, parallel=False, includeData=False,
                                showPlot=False, returnObject=True,
                            )
                            # In this native version validLib also defines prediction rows.
                            np.testing.assert_array_equal(model.idx_col, anchors)
                            np.testing.assert_array_equal(model.idx_tgt, anchors)
                        else:
                            model.Project(sample=1, seed=int(seed), parallel=False,
                                          includeData=False)
                        native = model.libMeans
                        assert int(native["LibSize"].iloc[0]) == library_size
                        values = native[["y:x", "x:y"]].iloc[0].to_numpy(dtype=float)
                        if not np.all(np.isfinite(values)) or np.any(np.abs(values) > 1 + 1e-12):
                            raise ValueError("Nonfinite or invalid native correlation.")
                        lag_rho[process_index, size_index, draw, lag_index, :] = values
                    except Exception as error:
                        errors.append({"process": process, "library_size": library_size,
                                       "draw": draw, "lag_steps": int(lag),
                                       "error": f"{type(error).__name__}: {error}"})
                        model = None
            elapsed = time.perf_counter() - started
            timings.append({"process": process, "library_size": library_size,
                            "seconds": elapsed})
            print(f"{process:11s} L={library_size:4d}: {elapsed:.1f} s", flush=True)

np.savez_compressed(
    OUTPUT_DIR / "lag_curves.npz", rho=lag_rho, processes=np.array(PROCESSES),
    library_sizes=np.array(LIBRARY_SIZES), library_seeds=library_seeds,
    lag_steps=LAG_STEPS, lag_years=lag_years, anchors=anchors,
    directions=np.array(DIRECTIONS),
)
error_table = pd.DataFrame(errors, columns=["process", "library_size", "draw", "lag_steps", "error"])
error_table.to_csv(OUTPUT_DIR / "failures.csv", index=False)
pd.DataFrame(timings).to_csv(OUTPUT_DIR / "timings.csv", index=False)
settings = {
    "inputs": [str((INPUT_DIR / f"r0_{p}_original_uniform_series.csv").relative_to(ROOT))
               for p in PROCESSES],
    "library_sizes": LIBRARY_SIZES, "n_libraries": N_LIBRARIES, "seed": LIBRARY_SEED,
    "E": E, "tau": TAU, "exclusion_radius": EXCLUSION_RADIUS, "dt_kyr": DT_KYR,
    "lag_steps": LAG_STEPS.tolist(), "common_anchor_count": len(anchors),
    "rho_dimensions": ["process", "library_size", "draw", "lag", "direction"],
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {p: version(p) for p in ["numpy", "scipy", "pandas", "pyEDM"]},
}
(OUTPUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Saved {np.isfinite(lag_rho).sum():,}/{lag_rho.size:,} finite rho values; "
      f"{len(errors)} failed native calls.")


## 4. Separate rho spread from peak-location spread
Classify each complete curve with the existing peak rule, including zero, boundary,
tied and failed results. SD and IQR describe all unique finite peaks, including
boundaries; status fractions always divide by the requested number of libraries.


In [ ]:
peak_rows = []
summary_rows = []
for process_index, process in enumerate(PROCESSES):
    for size_index, library_size in enumerate(LIBRARY_SIZES):
        for direction_index, direction in enumerate(DIRECTIONS):
            curves = lag_rho[process_index, size_index, :, :, direction_index]
            current_rows = []
            for draw, curve in enumerate(curves):
                peak_step, peak_rho, status = _ccm_lag_peak(curve, LAG_STEPS)
                row = {"process": process, "library_size": library_size,
                       "direction": direction, "draw": draw, "seed": int(library_seeds[draw]),
                       "peak_lag_years": peak_step * DT_KYR * 1000,
                       "peak_rho": peak_rho, "status": status}
                current_rows.append(row)
                peak_rows.append(row)
            group = pd.DataFrame(current_rows)
            unique_peaks = group.loc[group.peak_lag_years.notna(), "peak_lag_years"].to_numpy()
            complete_curves = curves[np.isfinite(curves).all(axis=1)]
            quantiles = np.percentile(unique_peaks, [5, 25, 50, 75, 95]) if len(unique_peaks) else np.full(5, np.nan)
            rho_band = (np.percentile(complete_curves, 95, axis=0)
                        - np.percentile(complete_curves, 5, axis=0)) if len(complete_curves) else np.full(len(LAG_STEPS), np.nan)
            summary = {
                "process": process, "library_size": library_size, "direction": direction,
                "requested": N_LIBRARIES, "complete_curves": len(complete_curves),
                "unique_peaks": len(unique_peaks),
                "peak_median_years": quantiles[2],
                "peak_sd_years": np.std(unique_peaks, ddof=1) if len(unique_peaks) > 1 else np.nan,
                "peak_iqr_years": quantiles[3] - quantiles[1],
                "peak_q05_years": quantiles[0], "peak_q95_years": quantiles[4],
                "mean_rho_q05_q95_width": np.mean(rho_band),
            }
            for status in ["past", "future", "zero", "boundary", "tied", "failed"]:
                count = int((group.status == status).sum())
                summary[f"n_{status}"] = count
                summary[f"fraction_{status}"] = count / N_LIBRARIES
            summary_rows.append(summary)

peaks = pd.DataFrame(peak_rows)
summary = pd.DataFrame(summary_rows)
peaks.to_csv(OUTPUT_DIR / "peak_lags.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(summary[["process", "direction", "library_size", "unique_peaks",
                 "peak_median_years", "peak_sd_years", "peak_iqr_years",
                 "mean_rho_q05_q95_width", "fraction_boundary", "fraction_tied",
                 "fraction_failed"]].round(4))


## 5. Compare rho curves and their random-library ranges
Lines show medians and shading shows pointwise 5th–95th percentiles across complete
library curves, not confidence intervals. Negative target shifts reconstruct the
past source; panels refer to causal hypotheses, without testing their significance.


In [ ]:
fig, axes = plt.subplots(len(PROCESSES), 2, figsize=(12, 9), sharex=True, layout="constrained")
for process_index, process in enumerate(PROCESSES):
    for direction_index, direction in enumerate(DIRECTIONS):
        ax = axes[process_index, direction_index]
        for size_index, (library_size, color) in enumerate(zip(LIBRARY_SIZES, COLORS)):
            curves = lag_rho[process_index, size_index, :, :, direction_index]
            complete = curves[np.isfinite(curves).all(axis=1)]
            if not len(complete):
                continue
            low, median, high = np.percentile(complete, [5, 50, 95], axis=0)
            ax.fill_between(lag_years, low, high, color=color, alpha=0.16)
            ax.plot(lag_years, median, color=color, label=f"L={library_size}")
        ax.axvline(0, color="0.5", linewidth=0.8, linestyle="--")
        ax.set_title(f"{process.capitalize()} | {direction}")
        ax.set_ylabel("Cross-map rho")
        ax.grid(alpha=0.15)
        if process_index == len(PROCESSES) - 1:
            ax.set_xlabel("Target shift Tp (years)")
axes[0, 0].legend(fontsize=9)
fig.suptitle("Median rho and 5–95% random-library ranges (panel-specific y scales)")
add_panel_labels(axes)
fig.savefig(FIGURE_DIR / "01_rho_by_library_length.png", dpi=180)
fig.savefig(FIGURE_DIR / "01_rho_by_library_length.pdf")
plt.show()


## 6. Inspect the distribution of optimal lags
Each point is the fraction of all requested libraries with a unique maximum at that
shift; lines only connect discrete bins. Boundary peaks remain visible, and tied or
failed curves have no assigned location, so a curve's frequencies may sum below one.


In [ ]:
fig, axes = plt.subplots(len(PROCESSES), 2, figsize=(12, 9), sharex=True, sharey=True, layout="constrained")
for process_index, process in enumerate(PROCESSES):
    for direction_index, direction in enumerate(DIRECTIONS):
        ax = axes[process_index, direction_index]
        for library_size, color in zip(LIBRARY_SIZES, COLORS):
            selected = peaks[(peaks.process == process) & (peaks.library_size == library_size)
                             & (peaks.direction == direction)]
            counts = np.array([(selected.peak_lag_years == lag).sum() for lag in lag_years])
            ax.plot(lag_years, counts / N_LIBRARIES, marker="o", markersize=3,
                    linewidth=1.1, color=color, label=f"L={library_size}")
        half_step = DT_KYR * 1000 / 2
        ax.axvspan(lag_years[0] - half_step, lag_years[0] + half_step, color="0.9", zorder=0)
        ax.axvspan(lag_years[-1] - half_step, lag_years[-1] + half_step, color="0.9", zorder=0)
        ax.axvline(0, color="0.5", linewidth=0.8, linestyle="--")
        ax.set_title(f"{process.capitalize()} | {direction}")
        ax.set_ylabel("Fraction of requested libraries")
        ax.set_ylim(0, 1.05)
        ax.set_xlim(lag_years[0] - half_step, lag_years[-1] + half_step)
        ax.grid(alpha=0.15)
        if process_index == len(PROCESSES) - 1:
            ax.set_xlabel("Optimal target shift (years)")
axes[0, 0].legend(fontsize=9)
fig.suptitle("Optimal lag distributions (gray bins mark search boundaries)")
add_panel_labels(axes)
fig.savefig(FIGURE_DIR / "02_peak_lag_distributions.png", dpi=180)
fig.savefig(FIGURE_DIR / "02_peak_lag_distributions.pdf")
plt.show()


## 7. Interpret this as a library-length diagnostic
A smaller library can widen rho ranges without moving the optimum, or can change
the preferred lag as well as its spread. Read SD/IQR together with the discrete
distribution and boundary/tie counts; a boundary concentration does not resolve an
optimum beyond the search window, and the peak need not equal the model's 20-year delay.

This three-length check is not a full CCM convergence study or a causal probability
calibration. It uses no chronology uncertainty and no independent-history repeats;
changing production library length would also require matching the original rho
and surrogate calculations to that length.

**Observed default run (2026-11-07).** All 55,800 rho values were finite. Reducing
L from 1119 to 350 increased the SD of the X -> Y optimum from 3.94 to 15.42 years
in the linear record, and from 3.43 to 12.71 years in the nonlinear record. Their
median optima stayed at -180 and -160 years, respectively. However, the nonlinear
Y -> X optimum had a larger SD at L=1119 (20.95 years) than at L=350 (16.04 years):
several competing peak locations make the response nonmonotonic. In the independent
Y -> X panel, 74% of the L=1119 peaks hit the -300-year boundary, so concentration
there does not establish a resolved lag or causation.

**Numerical check.** All 600 optimum locations and peak classifications at L=1119
matched the saved original baseline. Full rho arrays differed by at most
4.44e-16 because default CSV parsing slightly changes the originally generated
floating-point values. A first-draw check across every lag and both directions in
all three records reproduced the notebook exactly with its default parser, and
reproduced the original baseline exactly with round-trip parsing. The notebook
retains the existing prepared-input parsing convention for all library lengths.
